# 04 Predict with M8

Scores every held-out station with the M8 bundle trained for its fold in notebook 02.

Abbreviations used here: **RPF** is reverse power flow, the condition where a distribution substation exports power because rooftop solar exceeds local demand; a *wrong RPF sign* is a meter recording that stores the export as an import. **M7** is the deterministic threshold rule, **M8** the two-stage XGBoost classifier and **M9** the compact counterfactual method (revision 2). **MW** and **MWh** are megawatts and megawatt-hours; one interval is 15 minutes.

**Inputs.** The fold manifest, the bundle manifests `02_bundles/<fold_id>.json` and the two frozen datasets.

**Outputs.** `outputs/01_final_evaluation/04_m8/intervals_m8.parquet` (flagged slots, day flag, day and interval probabilities) and `manifests/04_m8_predict.json`.

**Approximate runtime.** About two minutes.

**Prerequisites.** Notebooks 01 and 02 (all eighteen bundles).

**Main process.**

1. Load the configuration and the folds.
2. For each held-out station, load its fold's bundle and run `pynrpf.api.run_inference` with the M8 configuration.
3. Write the interval prediction table.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display


def article_root() -> Path:
    """Locate publication/2_journal_article from JupyterLab, VS Code or the repository root."""
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "final_eval" / "cli.py").exists():
            return candidate
        nested = candidate / "publication" / "2_journal_article"
        if (nested / "final_eval" / "cli.py").exists():
            return nested
    raise FileNotFoundError("Could not locate publication/2_journal_article.")


ARTICLE = article_root()
sys.path.insert(0, str(ARTICLE))
sys.path.insert(0, str(ARTICLE.parents[1] / "src"))  # the repository's pynrpf package

from final_eval import cli, config  # noqa: E402

SETTINGS = config.load()  # verifies the frozen dataset hashes
OUT = SETTINGS.output_root()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Article root:", ARTICLE.relative_to(ARTICLE.parents[2]))

## 2. Predict

A missing bundle stops the run and names the fold; run notebook 02 for that fold first.

In [ ]:
table = cli.stage_predict(SETTINGS, "m8")
summary = (table.groupby(["cohort", "station"])
           .agg(days=("date", "nunique"), slots_flagged=("pred_interval", "sum"), mean_prob_day=("prob_day", "mean")).reset_index())
display(summary.round(3))

## Conclusion

The M8 interval table is written; notebook 06 derives its site-day outcomes at the frozen thresholds.